Chemins des fichiers récupérés (expression régulière) et des tables à créer

In [0]:
import org.apache.spark.sql.functions._
import com.databricks.dbutils_v1.DBUtilsHolder.dbutils
import org.apache.spark.sql.types._
import org.apache.spark.sql.DataFrame
import scala.collection.mutable.ListBuffer
//import org.apache.spark.sql.types.DataTypes
import org.apache.spark.sql.expressions.UserDefinedFunction
import scala.util.matching.Regex


// 🔧 Chemins physiques (personnalisables)
//val dbPath = "dbfs:/mnt/flight/flight_project.db"
//val tablePath = "dbfs:/mnt/flight" // /flight_clean"

val dbPath = "dbfs:/FileStore/flight_project.db"
val tablePath = "dbfs:/FileStore/flight" // /flight_clean"

val flightCsvPath ="dbfs:/FileStore/tables/flight/*.csv"
val weatherCsvPath ="dbfs:/FileStore/tables/weather/*.txt"
val airportTimezoneCsvPath ="dbfs:/FileStore/tables/referential/*.csv"


// Recrée la base si nécessaire (le metastore peut avoir disparu même si le dossier existe)
spark.sql(s"""
  CREATE DATABASE IF NOT EXISTS flight_project
  LOCATION '$dbPath'
""")



Création de la table Delta flight_raw.
Celle-ci contient les données brutes non typées
Si les fichiers de la tables sont présents on ne recharge pas csv. On n'exécute que la création de la table dans le cluster

In [0]:
val flightRawPath = tablePath+"/flight_raw"
// Vérifie si le dossier Delta existe dans DBFS
val deltaFlightRawExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == flightRawPath)


In [0]:


var flightDF: DataFrame = null
if (!deltaFlightRawExists) {
  flightDF = spark.read
    .option("header", "true")
    .option("inferSchema", "false") // force tout en StringType
    .csv(flightCsvPath)
}

In [0]:
// Sauvegarde en Delta dans un dossier explicite
if (!deltaFlightRawExists) {
  flightDF.write
    .format("delta")
    .mode("overwrite")
    .save(flightRawPath)
}

In [0]:
// Création de la table Delta externe (persistante) ==> doit être exécuter à chaque session pour récupérer la table dans l'environnement
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.flight_raw
  USING DELTA
  LOCATION '$flightRawPath'
""")

Création de la table airport_timezone_raw (même principe que plus haut)

In [0]:
val airportTimeZoneRawPath = tablePath+"/airport_timezone_raw"
// Vérifie si le dossier Delta existe dans DBFS
val deltaAirportTimeZoneRawExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == airportTimeZoneRawPath)


In [0]:
var airportTimezoneDF: DataFrame = null
if (!deltaAirportTimeZoneRawExists) {
  airportTimezoneDF = spark.read
    .option("header", "true")
    .option("inferSchema", "false") // force tout en StringType
    .csv(airportTimezoneCsvPath)
}

In [0]:

if (!deltaAirportTimeZoneRawExists) {
  airportTimezoneDF.write
    .format("delta")
    .mode("overwrite")
    .save(airportTimeZoneRawPath)
}


In [0]:
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.airport_timezone_raw
  USING DELTA
  LOCATION '$airportTimeZoneRawPath'
""")

Création de la table weather_raw (même principe que plus haut)

In [0]:
val weatherRawPath = tablePath+"/weather_raw"
// Vérifie si le dossier Delta existe dans DBFS
val deltaWeatherRawExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == weatherRawPath)

var weatherDF: DataFrame = null

if (!deltaWeatherRawExists) {
  weatherDF = spark.read
    .option("header", "true")
    .option("inferSchema", "false") // force tout en StringType
    .csv(weatherCsvPath)
}


In [0]:
if (!deltaWeatherRawExists) {
  weatherDF.write
    .format("delta")
    .mode("overwrite")
    .save(weatherRawPath)
}

In [0]:
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.weather_raw
  USING DELTA
  LOCATION '$weatherRawPath'
""")

In [0]:
%sql
CONVERT TO DELTA parquet.`dbfs:/user/hive/warehouse/flight_project.db/weather_raw`

Maintenant nous allons regarder airport_timezone

In [0]:

val dfAirport =spark.table("flight_project.airport_timezone_raw")
val constantCols = ListBuffer[String]()

dfAirport.columns.foreach { colName =>
  val colCounts = dfAirport.groupBy(col(colName)).count()
  val distinctCount = colCounts.count() // plus rapide que .distinct().count()

  if (distinctCount == 1) {
    constantCols += colName
  } else {
    println(s"=== Top 20 valeurs pour la colonne: $colName ===")
    colCounts.orderBy(desc("count")).show(20, truncate = false)
  }
}

println("=== Colonnes constantes (une seule valeur) ===")
constantCols.foreach(c => println(s" - $c"))

In [0]:


val checksAirports = Seq(
  dfAirport.filter(!$"AirportId".rlike("^[0-9]{5}$"))
    .select(col("AirportId").as("column_value"), lit("AirportId").as("column_name")),

  dfAirport.filter(!$"WBAN".rlike("^[0-9]{4,5}$"))
    .select(col("WBAN").as("column_value"), lit("WBAN").as("column_name")),
 
  dfAirport.filter(!$"TimeZone".rlike("^-\\d{1,2}$"))
    .select(col("TimeZone").as("column_value"), lit("TimeZone").as("column_name")),
)

// Union + groupBy + count
val invalidWithCountDFAirport = checksAirports.reduce(_ union _)
  .groupBy("column_name", "column_value")
  .count()
  .orderBy(col("column_name"), desc("count"))
// Affiche les résultats
invalidWithCountDFAirport.show(1000, truncate = false)

In [0]:
val airportTimeZoneCleanPath = tablePath + "/airport_timezone_clean"
//dbutils.fs.rm(flightCleanPath, recurse = true)
// Vérifie si le dossier Delta existe dans DBFS

val deltaAirportTimezoneCleanExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == airportTimeZoneCleanPath)

var cleanedAirportTimezoneDF: DataFrame = null

if (!deltaAirportTimezoneCleanExists) {
  println("Création du DataFrame nettoyé airport_timezone_clean car il n'existe pas")
  cleanedAirportTimezoneDF = dfAirport
    // Casts
    .withColumn("AirportId", $"AirportId".cast(ShortType))
    .withColumn("WBAN", $"WBAN".cast(IntegerType))
    .withColumn("TimeZone", $"TimeZone".cast(ByteType))

  cleanedAirportTimezoneDF.printSchema()
}

In [0]:
// Écrit sur un répertoire physique (meilleur contrôle)
if (!deltaAirportTimezoneCleanExists) {
  println("Création de la table delta")
  cleanedAirportTimezoneDF
    .coalesce(1) // Community Edition : évite d'exploser le disque
    .write
    .mode("overwrite")
    .format("delta")
    .save(airportTimeZoneCleanPath)
}


In [0]:
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.airport_time_zone_clean
  USING DELTA
  LOCATION '$airportTimeZoneCleanPath'
""")

Début de l'analyse des données de flight_raw

In [0]:
val dfFlightRaw = spark.table("flight_project.flight_raw")
val countRows = dfFlightRaw.count()
println(s"Nombre de lignes : $countRows")

In [0]:
val distinctDates = dfFlightRaw
  .select("FL_DATE")
  .distinct()

distinctDates.show(false) // false pour ne pas tronquer les dates longues

In [0]:


val result = dfFlightRaw.select(
  min(col("OP_CARRIER_AIRLINE_ID").cast("int")).alias("min_airline_id"),
  max(col("OP_CARRIER_AIRLINE_ID").cast("int")).alias("max_airline_id"),
  min(col("OP_CARRIER_FL_NUM").cast("int")).alias("min_flight_num"),
  max(col("OP_CARRIER_FL_NUM").cast("int")).alias("max_flight_num"),
  min(col("ORIGIN_AIRPORT_ID").cast("int")).alias("min_origin_id"),
  max(col("ORIGIN_AIRPORT_ID").cast("int")).alias("max_origin_id"),
  min(col("DEST_AIRPORT_ID").cast("int")).alias("min_dest_id"),
  max(col("DEST_AIRPORT_ID").cast("int")).alias("max_dest_id")
)

result.show(false)

Ici on s'assure que l'on a bien identifié le format de chaque colonne et qu'il n'y a pas de valeurs "hors format".
On va déjà supprimé les vols annulés ou déviés.

In [0]:
val dfFlightRawFiltered = dfFlightRaw.filter($"CANCELLED"=!="1.00").filter($"DIVERTED"=!="1.00")

val checks = Seq(
  dfFlightRawFiltered.filter($"FL_DATE".isNull || to_date($"FL_DATE", "yyyy-MM-dd").isNull)
    .select(col("FL_DATE").as("column_value"), lit("FL_DATE").as("column_name")),

  dfFlightRawFiltered.filter($"OP_CARRIER_AIRLINE_ID".isNull || !$"OP_CARRIER_AIRLINE_ID".rlike("^[0-9]{5}$"))
    .select(col("OP_CARRIER_AIRLINE_ID").as("column_value"), lit("OP_CARRIER_AIRLINE_ID").as("column_name")),

  dfFlightRawFiltered.filter($"OP_CARRIER_FL_NUM".isNull || !$"OP_CARRIER_FL_NUM".rlike("^[0-9]{1,4}$"))
    .select(col("OP_CARRIER_FL_NUM").as("column_value"), lit("OP_CARRIER_FL_NUM").as("column_name")),

  dfFlightRawFiltered.filter($"ORIGIN_AIRPORT_ID".isNull || !$"ORIGIN_AIRPORT_ID".rlike("^[0-9]{5}$"))
    .select(col("ORIGIN_AIRPORT_ID").as("column_value"), lit("ORIGIN_AIRPORT_ID").as("column_name")),

  dfFlightRawFiltered.filter($"DEST_AIRPORT_ID".isNull || !$"DEST_AIRPORT_ID".rlike("^[0-9]{5}$"))
    .select(col("DEST_AIRPORT_ID").as("column_value"), lit("DEST_AIRPORT_ID").as("column_name")),

  dfFlightRawFiltered.filter($"CRS_DEP_TIME".isNull || !$"CRS_DEP_TIME".rlike("^(?:[01][0-9]|2[0-3])[0-5][0-9]$"))
    .select(col("CRS_DEP_TIME").as("column_value"), lit("CRS_DEP_TIME").as("column_name")),

  dfFlightRawFiltered.filter($"ARR_DELAY_NEW".isNull || !$"ARR_DELAY_NEW".rlike("^\\d{1,4}\\.\\d{2}$"))
    .select(col("ARR_DELAY_NEW").as("column_value"), lit("ARR_DELAY_NEW").as("column_name")),

  dfFlightRawFiltered.filter($"CRS_ELAPSED_TIME".isNull || !$"CRS_ELAPSED_TIME".rlike("^\\d{1,4}\\.\\d{2}$"))
    .select(col("CRS_ELAPSED_TIME").as("column_value"), lit("CRS_ELAPSED_TIME").as("column_name")),

  dfFlightRawFiltered.filter($"WEATHER_DELAY".isNull || !$"WEATHER_DELAY".rlike("^\\d{1,4}\\.\\d{2}$"))
    .select(col("WEATHER_DELAY").as("column_value"), lit("WEATHER_DELAY").as("column_name")),

  dfFlightRawFiltered.filter($"NAS_DELAY".isNull || !$"NAS_DELAY".rlike("^\\d{1,4}\\.\\d{2}$"))
    .select(col("NAS_DELAY").as("column_value"), lit("NAS_DELAY").as("column_name")),

  dfFlightRawFiltered.filter($"CANCELLED".isNull || !($"CANCELLED" === "0.00" || $"CANCELLED" === "1.00"))
    .select(col("CANCELLED").as("column_value"), lit("CANCELLED").as("column_name")),

  dfFlightRawFiltered.filter($"DIVERTED".isNull || !($"DIVERTED" === "0.00" || $"DIVERTED" === "1.00"))
    .select(col("DIVERTED").as("column_value"), lit("DIVERTED").as("column_name"))
)

// Union + groupBy + count
val invalidWithCountDF = checks.reduce(_ union _)
  .groupBy("column_name", "column_value")
  .count()
  .orderBy(col("column_name"), desc("count"))
// Affiche les résultats
invalidWithCountDF.show(1000, truncate = false)

Les lignes avec CRS_DEP_TIME ou CRS_ELAPSED_TIME ou ARR_DELAY_NEW vides sont a priori inexploitables et seront supprimées.
Pour les autres, nous mettrons 0 lors du retraitement

In [0]:
val distinctC12 = dfFlightRawFiltered
  .select("_c12")
  .distinct()

distinctC12.show(false)

La colonne _c12 est constante et sera supprimée

Maintenant on va vérifier que tous les aéroports de flight sont bien dans la table de correspondance airportID/WBAN

In [0]:
val refs = dfAirport.select(
  min(col("AirportID").cast("int")).alias("AirportID"),
  max(col("AirportID").cast("int")).alias("AirportID"),
  min(col("WBAN").cast("int")).alias("WBAN"),
  max(col("WBAN").cast("int")).alias("WBAN")
)

refs.show(false)

// Création d'une colonne unique combinant les deux colonnes
val airportIDs = dfFlightRawFiltered
  .select(col("ORIGIN_AIRPORT_ID").cast(ShortType).as("AIRPORT_ID"))
  .union(dfFlightRawFiltered.select(col("DEST_AIRPORT_ID").cast(ShortType).as("AIRPORT_ID")))

// Calcul du min et du max
val result = airportIDs.agg(
  min("AIRPORT_ID").as("min_airport_id"),
  max("AIRPORT_ID").as("max_airport_id")
)

result.show(false)


In [0]:
val dfAirport = spark.read.format("delta").load(tablePath + "/airport_timezone_clean")

val allFlightAirportIds = dfFlightRawFiltered
  .select(col("ORIGIN_AIRPORT_ID").cast(ShortType).alias("AirportId"))
  .union(dfFlightRawFiltered.select(col("DEST_AIRPORT_ID").cast(ShortType).alias("AirportId")))
  .distinct()

val missingAirportIds = allFlightAirportIds
  .join(dfAirport.select("AirportId").distinct(), Seq("AirportId"), "left_anti")

val unknownAirportIdsDF = missingAirportIds.cache()

// Rejoindre avec cast sur les colonnes nécessaires
val flightsWithUnknownOrigin = dfFlightRawFiltered
  .withColumn("ORIGIN_AIRPORT_ID_SHORT", col("ORIGIN_AIRPORT_ID").cast(ShortType))
  .join(missingAirportIds, col("ORIGIN_AIRPORT_ID_SHORT") === col("AirportId"))
  .groupBy("ORIGIN_AIRPORT_ID").count()

val flightsWithUnknownDest = dfFlightRawFiltered
  .withColumn("DEST_AIRPORT_ID_SHORT", col("DEST_AIRPORT_ID").cast(ShortType))
  .join(missingAirportIds, col("DEST_AIRPORT_ID_SHORT") === col("AirportId"))
  .groupBy("DEST_AIRPORT_ID").count()

flightsWithUnknownOrigin.show(1000, false)
flightsWithUnknownDest.show(1000, false)

// Pour les lignes complètes
val flightsWithMissingOrigin = dfFlightRawFiltered
  .withColumn("ORIGIN_AIRPORT_ID_SHORT", col("ORIGIN_AIRPORT_ID").cast(ShortType))
  .join(missingAirportIds, col("ORIGIN_AIRPORT_ID_SHORT") === col("AirportId"))

val flightsWithMissingDest = dfFlightRawFiltered
  .withColumn("DEST_AIRPORT_ID_SHORT", col("DEST_AIRPORT_ID").cast(ShortType))
  .join(missingAirportIds, col("DEST_AIRPORT_ID_SHORT") === col("AirportId"))

val totalMissingFlights = flightsWithMissingOrigin
  .union(flightsWithMissingDest)
  .dropDuplicates()

println(s"Nombre total de lignes concernées : ${totalMissingFlights.count()}")

Création du dataframe pour la table flight_clean. La table delta flight_clean sera persistante. Donc la construction du dataframe ne sera réalisée que si les fichiers de la table Delta n'existent pas dans l'environnement.
Si on veut forcer la recréation de la table avec de nouvelles données il faut décommenter la ligne 


dbutils.fs.rm(flightCleanPath, recurse = true)

In [0]:
//val rawDF = spark.table("flight_project.flight_raw")
val flightCleanPath = tablePath + "/flight_clean"
//dbutils.fs.rm(flightCleanPath, recurse = true)
// Vérifie si le dossier Delta existe dans DBFS

val deltaFlightCleanExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == flightCleanPath)

var cleanedFlightDF: DataFrame = null

if (!deltaFlightCleanExists) {
  println("Création du DataFrame nettoyé flight_clean car il n'existe pas")
    // Convertit les airportId inconnus en String pour comparaison
  val unknownAirportIdsStrDF = unknownAirportIdsDF
    .withColumn("AirportIdStr", col("AirportId").cast(StringType))
    .select("AirportIdStr")

  // Exclusion des lignes avec aéroports inconnus
  val filteredDF = dfFlightRawFiltered
    .join(unknownAirportIdsStrDF, dfFlightRawFiltered("ORIGIN_AIRPORT_ID") === unknownAirportIdsStrDF("AirportIdStr"), "left_anti")
    .join(unknownAirportIdsStrDF, dfFlightRawFiltered("DEST_AIRPORT_ID") === unknownAirportIdsStrDF("AirportIdStr"), "left_anti")

  // Nettoyage des colonnes
  cleanedFlightDF = filteredDF
    // Exclusion des lignes critiques avec valeurs nulles
    .filter($"ARR_DELAY_NEW".isNotNull &&
            $"CRS_ELAPSED_TIME".isNotNull &&
            $"CRS_DEP_TIME".isNotNull)
    
    // Casts et nettoyages
    .withColumn("FL_DATE", to_date($"FL_DATE", "yyyy-MM-dd"))
    .withColumn("OP_CARRIER_AIRLINE_ID", $"OP_CARRIER_AIRLINE_ID".cast(ShortType))
    .withColumn("OP_CARRIER_FL_NUM", $"OP_CARRIER_FL_NUM".cast(ShortType))
    .withColumn("ORIGIN_AIRPORT_ID", $"ORIGIN_AIRPORT_ID".cast(ShortType))
    .withColumn("DEST_AIRPORT_ID", $"DEST_AIRPORT_ID".cast(ShortType))
    .withColumn("ARR_DELAY_NEW", $"ARR_DELAY_NEW".cast(FloatType))
    .withColumn("CRS_ELAPSED_TIME", $"CRS_ELAPSED_TIME".cast(FloatType))

    // WEATHER_DELAY : remplacement des nulls par 0 + indicateur
    .withColumn("HAS_WEATHER_DELAY", $"WEATHER_DELAY".isNotNull)
    .withColumn("WEATHER_DELAY", coalesce($"WEATHER_DELAY".cast(FloatType), lit(BigDecimal(0.0)).cast(FloatType)))

    // NAS_DELAY : idem
    .withColumn("HAS_NAS_DELAY", $"NAS_DELAY".isNotNull)
    .withColumn("NAS_DELAY", coalesce($"NAS_DELAY".cast(FloatType), lit(BigDecimal(0.0)).cast(FloatType)))

    // Décomposition de CRS_DEP_TIME
    .withColumn("CRS_DEP_TIME_STR", lpad($"CRS_DEP_TIME".cast(StringType), 4, "0"))
    .withColumn("CRS_DEP_TIME_INT", $"CRS_DEP_TIME_STR".cast(IntegerType))
    .withColumn("CRS_DEP_TIMESTAMP", to_timestamp(concat($"FL_DATE", $"CRS_DEP_TIME_STR"), "yyyy-MM-ddHHmm"))

    // Supprime colonne indésirable
    .drop("_c12")
    .drop("CANCELLED")
    .drop("DIVERTED")

  cleanedFlightDF.printSchema()
}

In [0]:

// Écrit sur un répertoire physique (meilleur contrôle)
if (!deltaFlightCleanExists) {
  println("Création de la table delta")

  cleanedFlightDF
    .coalesce(1) // Community Edition : évite d'exploser le disque
    .write
    .mode("overwrite")
    .format("delta")
    .save(flightCleanPath)
}


Création de la table delta dans l'environnement (à exécuter à chaque session)

In [0]:
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.flight_clean
  USING DELTA
  LOCATION '$flightCleanPath'
""")

Maintenant, nous nous occupons de la table weather_clean

In [0]:
val dfWeather = spark.table("flight_project.weather_raw")

val result = dfWeather
  .select(
    min(col("wban").cast("int")).alias("min_wban"),
    max(col("wban").cast("int")).alias("max_wban")
  )

result.show(false)

Pour chaque colonne, nous affichons les 20 valeurs les plus fréquentes.
Nous listons également les colonnes constantes qui pourront être supprimées

In [0]:


val constantCols = ListBuffer[String]()

dfWeather.columns.foreach { colName =>
  val colCounts = dfWeather.groupBy(col(colName)).count()
  val distinctCount = colCounts.count() // plus rapide que .distinct().count()

  if (distinctCount == 1) {
    constantCols += colName
  } else {
    println(s"=== Top 20 valeurs pour la colonne: $colName ===")
    colCounts.orderBy(desc("count")).show(20, truncate = false)
  }
}

println("=== Colonnes constantes (une seule valeur) ===")
constantCols.foreach(c => println(s" - $c"))

Suite à cette analyse, nous allons nous assurer que nous avons bien identifié le format de chaque colonne en appliquant une expression régulière. Si des lignes ressortes, c'est soit que notre analyse n'est pas bonne ou incomplète, soit qu'il y a des valeurs aberrantes.
A priori nous avons des données en Celsius et en Faranheit. Elles font donc doublon.
Nous allons préférer les colonnes en Faranheit car il y a moins de valeurs "s" dans les flags Faranheit que dans celles Celsius.

Les colonnes WeatherType et SkyCondition seront traitées à part car plus complexes

In [0]:
val dfWeatherRaw = spark.table("flight_project.weather_raw")
// 1) Exclusion des colonnes en Celsius
val keptCols: Array[String] = dfWeatherRaw.columns.filterNot(_.contains("Celsius"))

// 2) Sélectionner uniquement ces colonnes
val df = dfWeatherRaw.select(keptCols.map(col): _*)

val checks = Seq(

  df.filter(col("WBAN").isNull || trim(col("WBAN")) === "" ||
    !trim(col("WBAN")).rlike("^[0-9]{1,5}$"))
    .select(col("WBAN").alias("column_value"), lit("WBAN").alias("column_name")),
  // Vérification de date : null ou format non yyyyMMdd
  df.filter(col("date").isNull || to_date(col("date"), "yyyyMMdd").isNull)
    .select(col("date").alias("column_value"), lit("DATE").alias("column_name")),

  // Vérification time : HHmm (00–23)(00–59)
  //Toujours 4 caractères + 2 premiers caractères entre 00 et 23 (entre 00 et 19 : [01][0-9] ou entre 20 et 23 : 2[0-3]) 
  //+ 2 derniers caratères entre 00 et 59 [0-5][0-9]
  df.filter(col("time").isNull || !col("time").rlike("^(?:[01][0-9]|2[0-3])[0-5][0-9]$"))
    .select(col("time").alias("column_value"), lit("TIME").alias("column_name")),

  // Vérification flags (non vides et différents de "s")
  df.filter(trim(col("SkyConditionFlag")) =!= "" && col("SkyConditionFlag") =!= "s")
    .select(col("SkyConditionFlag").alias("column_value"), lit("SkyConditionFlag").alias("column_name")),

  df.filter(trim(col("VisibilityFlag")) =!= "" && col("VisibilityFlag") =!= "s")
    .select(col("VisibilityFlag").alias("column_value"), lit("VisibilityFlag").alias("column_name")),

  df.filter(trim(col("WeatherTypeFlag")) =!= "" && col("WeatherTypeFlag") =!= "s")
    .select(col("WeatherTypeFlag").alias("column_value"), lit("WeatherTypeFlag").alias("column_name")),

//  df.filter(trim(col("DryBulbCelsiusFlag")) =!= "" && col("DryBulbCelsiusFlag") =!= "s")
//    .select(col("DryBulbCelsiusFlag").alias("column_value"), lit("DryBulbCelsiusFlag").alias("column_name")),

//  df.filter(trim(col("DewPointCelsiusFlag")) =!= "" && col("DewPointCelsiusFlag") =!= "s")
//    .select(col("DewPointCelsiusFlag").alias("column_value"), lit("DewPointCelsiusFlag").alias("column_name")),

  df.filter(trim(col("WindSpeedFlag")) =!= "" && col("WindSpeedFlag") =!= "s")
    .select(col("WindSpeedFlag").alias("column_value"), lit("WindSpeedFlag").alias("column_name")),

  df.filter(trim(col("WindDirectionFlag")) =!= "" && col("WindDirectionFlag") =!= "s")
    .select(col("WindDirectionFlag").alias("column_value"), lit("WindDirectionFlag").alias("column_name")),

  df.filter(trim(col("ValueForWindCharacterFlag")) =!= "" && col("ValueForWindCharacterFlag") =!= "s")
    .select(col("ValueForWindCharacterFlag").alias("column_value"), lit("ValueForWindCharacterFlag").alias("column_name")),

  df.filter(trim(col("SeaLevelPressureFlag")) =!= "" && col("SeaLevelPressureFlag") =!= "s")
    .select(col("SeaLevelPressureFlag").alias("column_value"), lit("SeaLevelPressureFlag").alias("column_name")),

  df.filter(trim(col("HourlyPrecipFlag")) =!= "" && col("HourlyPrecipFlag") =!= "s")
    .select(col("HourlyPrecipFlag").alias("column_value"), lit("HourlyPrecipFlag").alias("column_name")),

  df.filter(trim(col("AltimeterFlag")) =!= "" && col("AltimeterFlag") =!= "s")
    .select(col("AltimeterFlag").alias("column_value"), lit("AltimeterFlag").alias("column_name")),

  // Températures avec décimales (M est valide)
  //df.filter(col("DryBulbCelsius") =!= "M" && !col("DryBulbCelsius").rlike("^[-]?\\d{1,2}\\.\\d$"))
  //  .select(col("DryBulbCelsius").alias("column_value"), lit("DryBulbCelsius").alias("column_name")),

  //df.filter(col("WetBulbCelsius") =!= "M" && !col("WetBulbCelsius").rlike("^[-]?\\d{1,2}\\.\\d$"))
  //  .select(col("WetBulbCelsius").alias("column_value"), lit("WetBulbCelsius").alias("column_name")),

  //df.filter(col("DewPointCelsius") =!= "M" && !col("DewPointCelsius").rlike("^[-]?\\d{1,2}\\.\\d$"))
  //  .select(col("DewPointCelsius").alias("column_value"), lit("DewPointCelsius").alias("column_name")),

  df.filter(col("DryBulbFarenheit") =!= "M" && !trim(col("DryBulbFarenheit")).rlike("^-?\\d{1,3}$"))
    .select(col("DryBulbFarenheit").alias("column_value"), lit("DryBulbFarenheit").alias("column_name")),

  df.filter(col("WetBulbFarenheit") =!= "M" && !col("WetBulbFarenheit").rlike("^-?\\d{1,2}$"))
    .select(col("WetBulbFarenheit").alias("column_value"), lit("WetBulbFarenheit").alias("column_name")),

  df.filter(col("DewPointFarenheit") =!= "M" && !trim(col("DewPointFarenheit")).rlike("^-?\\d{1,3}$"))
    .select(col("DewPointFarenheit").alias("column_value"), lit("DewPointFarenheit").alias("column_name")),

  df.filter(col("RelativeHumidity") =!= "M" && !trim(col("RelativeHumidity")).rlike("^\\d{1,3}$"))
    .select(col("RelativeHumidity").alias("column_value"), lit("RelativeHumidity").alias("column_name")),

  df.filter(//trim(col("WindSpeed")) =!= "" && 
    col("WindSpeed") =!= "M" &&
    !trim(col("WindSpeed")).rlike("^\\d{1,3}$"))
    .select(col("WindSpeed").alias("column_value"), lit("WindSpeed").alias("column_name")),

  df.filter(//trim(col("WindDirection")) =!= "" && 
    col("WindDirection") =!= "M" &&
    trim(col("WindDirection")) =!= "VR" &&
    !col("WindDirection").rlike("^\\d{1,3}$"))
    .select(col("WindDirection").alias("column_value"), lit("WindDirection").alias("column_name")),

  df.filter(trim(col("ValueForWindCharacter")) =!= "" &&
    !col("ValueForWindCharacter").rlike("^\\d{1,3}$"))
    .select(col("ValueForWindCharacter").alias("column_value"), lit("ValueForWindCharacter").alias("column_name")),

  df.filter(col("StationPressure") =!= "M" &&
    !col("StationPressure").rlike("^\\d{1,2}\\.\\d{2}$"))
    .select(col("StationPressure").alias("column_value"), lit("StationPressure").alias("column_name")),

  df.filter(trim(col("PressureTendency")) =!= "" &&
    !col("PressureTendency").rlike("^[0-8]$"))
    .select(col("PressureTendency").alias("column_value"), lit("PressureTendency").alias("column_name")),

  df.filter(trim(col("PressureChange")) =!= "" &&
    !col("PressureChange").rlike("^\\d{1,3}$"))
    .select(col("PressureChange").alias("column_value"), lit("PressureChange").alias("column_name")),

  df.filter(col("SeaLevelPressure") =!= "M" &&
    !col("SeaLevelPressure").rlike("^\\d{1,2}\\.\\d{2}$"))
    .select(col("SeaLevelPressure").alias("column_value"), lit("SeaLevelPressure").alias("column_name")),

  df.filter(trim(col("HourlyPrecip")) =!= "" && trim(col("HourlyPrecip")) =!= "T" &&
    !trim(col("HourlyPrecip")).rlike("^\\d{1,2}\\.\\d{2}$"))
    .select(col("HourlyPrecip").alias("column_value"), lit("HourlyPrecip").alias("column_name")),

  df.filter(col("Altimeter") =!= "M" &&
    !col("Altimeter").rlike("^\\d{1,2}\\.\\d{2}$"))
    .select(col("Altimeter").alias("column_value"), lit("Altimeter").alias("column_name"))
)

// Agrégation des erreurs
val invalidWeatherDF = checks.reduce(_ union _)
  .groupBy("column_name", "column_value")
  .count()
  .orderBy(col("column_name"), desc("count"))

invalidWeatherDF.show(1000, truncate = false)


Il y a 2 valeurs aberrantes dans HourlyPrecip. On va regarder si l'on voit d'autres problèmes dans les lignes concernées
Les valeurs manquantes de WindDirection et WindSpeed sont censées être valorisée par "M". Or on constate qu'il y a quelques valeurs vides. Nous les traiterons comme des "M"

In [0]:
%sql
SELECT *
FROM flight_project.weather_raw
WHERE 
  trim(HourlyPrecip) != '' and
  trim(HourlyPrecip) != 'T' and
  NOT REGEXP_LIKE(trim(HourlyPrecip), '^\\d{1,2}\\.\\d{2}$')

Pas de problème flagrant ==> dans le retraitement nous mettrons arbitrairement HourlyPrecip à 0

Maintenant nous allons regarder le format de SkyCondition

In [0]:
val allSkyConditions = df.select("SkyCondition")
  .flatMap(row => Option(row.getString(0)).getOrElse("").split(" "))
  .distinct
  .collect
  .sorted

Il semble que la colonne soit composées d'un ensemble de séquences, elles mêmes composées de 1 à 3 parties. La 1ère est consituée de 2 ou 3 lettres, la 2ème (non présente sur certaines) de 3 chiffres et la dernière (souvent non présente) de 2 ou 3 lettre.
Il y a également "M" qui est présent, représentant l'absence de données.
Nous allons vérifier qu'il n'y a pas de valeurs en dehors de ce format

In [0]:
%sql
SELECT distinct SkyCondition
FROM flight_project.weather_raw
WHERE SkyCondition !='M'
  AND NOT trim(SkyCondition) RLIKE '^([A-Z]{2,3}([0-9]{3})?([A-Z]{2,3})?)( ([A-Z]{2,3}([0-9]{3})?([A-Z]{2,3})?))*$'

On remarque que le format et bon et qu'il n'y a pas de valeurs nulle ou vide.
Maintenant nous allons afficher les valeurs possibles des 3 parties des séquences

In [0]:
val skyPattern = "([A-Z]{2,3})([0-9]{3})?([A-Z]{2,3})?$"

// Étape 1 : explode sur les séquences (on suppose séparées par des espaces)
val exploded = dfWeather
  .filter(col("SkyCondition").isNotNull)
  .withColumn("sequence", explode(split(col("SkyCondition"), " ")))
  .filter(col("sequence").rlike(skyPattern))


// Étape 2 : extraire le préfixe à la fin de chaque séquence
val prefixesDF = exploded
  .withColumn("prefix", regexp_extract(col("sequence"), skyPattern, 1))
  .filter(length(col("prefix")) > 0)
  .select("prefix")
  .distinct()
prefixesDF.show(false)

// Étape  3 : extraire les valeurs
val valuesDF = exploded
  .withColumn("value", regexp_extract(col("sequence"), skyPattern, 2))
  .filter(length(col("value")) > 0)
  .select("value")
  .distinct()

valuesDF.show(false)


// Étape 4 : extraire le suffixe à la fin de chaque séquence, s’il existe
val suffixesDF = exploded
  .withColumn("suffix", regexp_extract(col("sequence"), skyPattern, 3))
  .filter(length(col("suffix")) > 0)
  .select("suffix")
  .distinct()

suffixesDF.show(false)

Maintenant nous allons regarder la répartition par nombre de séquences.

In [0]:


// UDF pour compter le nombre de séquences (0 si "M" ou vide)
val countSequences: UserDefinedFunction = udf { skyCond: String =>
  if (skyCond == null || skyCond.trim.isEmpty || skyCond.trim == "M") 0
  else skyCond.trim.split(" ").length
}

// Application + GroupBy
val seqCountDF = dfWeather
  .withColumn("nb_sequences", countSequences(col("SkyCondition")))
  .groupBy("nb_sequences")
  .count()
  .orderBy(desc("nb_sequences"))

// Affichage
seqCountDF.show(false)

Maintenant nous allons regarder la colonne WeatherType

In [0]:
val allWeatherTypes = df.select("SkyCondition")
  .flatMap(row => Option(row.getString(0)).getOrElse("").split(" "))
  .distinct
  .collect
  .sorted

La colonne semble elle aussi constituées de plusieurs séquence. Chacune de ces séquences sont constituée éventuellement d'un + ou - au début puis d'une série de couple de lettres. 

In [0]:
%sql
SELECT WeatherType
FROM flight_project.weather_raw
WHERE trim(WeatherType) !=''
  AND NOT trim(WeatherType) RLIKE '^([+-]?([A-Z]{2})+)( [+-]?([A-Z]{2})+)*$'

Il y a 2 types de séquences aberrantes : TS+RA => le plus probable est un espace manquant qui sera rajouté dans le retraitement

                                          -+FC => nous le retraiterons arbitrairement en +FC

Maintenant nous allons regarder les couples de lettres présents

In [0]:
// UDF pour extraire tous les groupes de 2 lettres majuscules
val extractLetterPairs = udf { (wt: String) =>
  if (wt == null) Seq.empty[String]
  else {
    val pattern = "([A-Z]{2})".r
    pattern.findAllIn(wt).toSeq.distinct
  }
}

// Appliquer l’UDF et exploser
val letterPairsDF = dfWeather
  .withColumn("letter_pairs", extractLetterPairs(col("WeatherType")))
  .select(explode(col("letter_pairs")).as("pair"))
  .distinct()
  .orderBy("pair")

letterPairsDF.show(1000,false)

In [0]:
// UDF pour compter le nombre de séquences (0 si "M" ou vide)
//val countSequences: UserDefinedFunction = udf { skyCond: String =>
//  if (skyCond == null || skyCond.trim.isEmpty || skyCond.trim == "M") 0
//  else skyCond.trim.split(" ").length
//}

// Application + GroupBy
val seqCountDF = dfWeather
  .withColumn("nb_sequences", countSequences(col("WeatherType")))
  .groupBy("nb_sequences")
  .count()
  .orderBy(desc("nb_sequences"))

// Affichage
seqCountDF.show(false)

Maintenant, nous allons faire le retraitement et la création de weather_clean

Pour commencer, nous allons préparer des fonctions qui nous permettrons d'enrichir weather_clean avec d'autres colonnes liées à SkyCondition

==> le nombre de séquences

==> l'altitude de la 1ère couche (calculée à l'aide d'un colonne temporaire sky_altitude)

==> l'altitude de la dernière couche (calculée à l'aide d'un colonne temporaire sky_altitude)

==> la moyenne des couches (calculée à l'aide d'un colonne temporaire sky_altitude)

==> La présence ou non de CB

==> La présence ou non de TCU

==> La présence ou non de chacun des préfixes (une colonne booléenne par préfixe)

In [0]:
// UDF pour compter le nombre de séquences
val countSeq: UserDefinedFunction = udf { sc: String =>
  if (sc == null || sc.trim == "M" || sc.trim == "CLR") 0
  else sc.trim.split(" ").length
}

// UDF pour détecter la présence de CB
val hasCB: UserDefinedFunction = udf((sc: String) => Option(sc).exists(_.contains("CB")))

// UDF pour détecter la présence de TCU
val hasTCU: UserDefinedFunction = udf((sc: String) => Option(sc).exists(_.contains("TCU")))

// UDF pour extraire toutes les altitudes valides (3 chiffres après 2 ou 3 lettres)
val extractAltitudes: UserDefinedFunction = udf { sc: String =>
  if (sc == null || sc.trim == "M") Seq.empty[Int]
  else {
    val pattern = "([A-Z]{2,3})(\\d{3})([A-Z]{2,3})?".r
    pattern.findAllMatchIn(sc).map(m => m.group(2).toInt).toSeq
  }
}

// Liste des préfixes connus
val skyPrefixes = Seq("BKN", "CLR", "FEW", "OVC", "SCT", "VV")


Nous allons filtrer les lignes qui sont dans les aéroports présents dans flight_clean. Pour cela nous récupérons la liste des WBAN concernés.


In [0]:
val dfFlight = spark.read.format("delta").load(tablePath + "/flight_clean")
val dfAirport = spark.read.format("delta").load(tablePath + "/airport_timezone_clean")

// Étape 1 : Récupérer les AirportId utilisés dans flight_clean (origin et dest)
val flightAirportIds = dfFlight
  .select($"ORIGIN_AIRPORT_ID".alias("AirportId"))
  .union(dfFlight.select($"DEST_AIRPORT_ID".alias("AirportId")))
  .distinct()

// Étape 2 : Faire la jointure avec dfAirport pour récupérer les WBAN
val airportIdWithWbanDF = flightAirportIds
  .join(dfAirport.select($"AirportId", $"WBAN"), Seq("AirportId"), "inner")
  .distinct()

// Optionnel : affichage ou usage
airportIdWithWbanDF.show(false)

Pour WeatherType nous allons faire une colonne par couple de lettre existant. La valeur sera

=> 0 si le couple n'est pas présent

=> 1 si le couple est présent précédé d'un "-"

=> 2 si le couple est présent et n'est pas précédé d'un "-" ni d'un "+"

=> 4 si le couple est présent et précédé d'un "+"

In [0]:
//val allPairs = letterPairsDF.select("pair").as[String].collect().toSeq
val weatherPairs: Seq[String] = letterPairsDF.select("pair").as[String].collect().toSeq

val extractWeatherScores = udf { wt: String =>
  val pattern = """([+-]?)([A-Z]{2,})""".r
  val result = scala.collection.mutable.Map[String, Byte]()

  if (wt != null) {
    for (m <- pattern.findAllMatchIn(wt)) {
      val prefix = m.group(1)
      val letters = m.group(2)
      val score: Byte = prefix match {
        case "+" => 4.toByte
        case "-" => 1.toByte
        case _   => 2.toByte
      }

      letters.grouped(2).foreach { pair =>
        result(pair) = Math.max(result.getOrElse(pair, 0.toByte), score).toByte
      }
    }
  }

  result.toMap
}

Maintenant nous allons pouvoir créer le dataframe cible retraité qui permettra de créer la table Delta weather_clean avec les transformations, corrections listées plus haut


In [0]:
// Étape 0 : Chargement
val rawDF = spark.table("flight_project.weather_raw")
val rawDfWoCelsius = rawDF.select(keptCols.map(col): _*)
val weatherCleanPath = tablePath + "/weather_clean"
dbutils.fs.rm(weatherCleanPath, recurse = true)

// Vérifie si le dossier Delta existe dans DBFS
val deltaWeatherCleanExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == weatherCleanPath)

var weather_clean: DataFrame = null

if (!deltaWeatherCleanExists) {
  println("Création du DataFrame nettoyé weather_clean car il n'existe pas")

  // Étape 1 : Correction de WeatherType
  val correctedWeather = rawDfWoCelsius.withColumn(
    "WeatherType",
    when(trim(col("WeatherType")) === "+FC TS+RA", "+FC TS +RA")
      .when(trim(col("WeatherType")) === "-+FC TSRA", "+FC TS RA")
      .otherwise(col("WeatherType"))
  )

  // Étape 2 : Date et heure
  val withTime = correctedWeather
    .withColumn("date", to_date(col("date"), "yyyyMMdd"))
    .withColumn("hour", substring(col("time"), 1, 2).cast("byte"))
    .withColumn("minute", substring(col("time"), 3, 2).cast("byte"))
  val withTimestamp = withTime.withColumn(
    "timestamp",
    to_timestamp(
      concat_ws(" ", col("date"), lpad(col("hour").cast("string"), 2, "0"), lpad(col("minute").cast("string"), 2, "0")),
      "yyyy-MM-dd HH mm"
    )
  )
  // Étape 3 : Nettoyage des flags ("" et "s" → null)
  val flagCols = Seq(
    "SkyConditionFlag", "VisibilityFlag", "WeatherTypeFlag",
    //"DryBulbCelsiusFlag", "DewPointCelsiusFlag",
    "WindSpeedFlag", "WindDirectionFlag",
    "ValueForWindCharacterFlag", "SeaLevelPressureFlag", "HourlyPrecipFlag", "AltimeterFlag"
  )

  val cleanedFlags = flagCols.foldLeft(withTimestamp) { (df, colName) =>
    df.withColumn(colName, when(trim(col(colName)) === "" || col(colName) === "s", lit(null)).otherwise(col(colName)))
  }

  // Étape 3 bis : conversion flags → booléen (estimation ?)
  val flagsWithBooleans = flagCols.foldLeft(cleanedFlags) { (df, colName) =>
    df.withColumn(s"${colName}_bool", col(colName).isNotNull)
  }.drop(flagCols: _*) // On peut supprimer les versions originales

  // Étape 4 : SkyCondition → features
  val countSeq = udf((sc: String) => if (sc == null || sc.trim == "M") 0 else sc.trim.split(" ").length)
  val hasCB = udf((sc: String) => Option(sc).exists(_.contains("CB")))
  val hasTCU = udf((sc: String) => Option(sc).exists(_.contains("TCU")))

// Ajout des indicateurs pour chaque préfixe
  val withSkyIndicators = skyPrefixes.foldLeft(flagsWithBooleans) { (df, prefix) =>
    df.withColumn(s"sky_has_$prefix", col("SkyCondition").contains(prefix))
  }

  // Application sur DataFrame
  val withSkyDerived = withSkyIndicators
    .withColumn("sky_num_layers", countSeq(col("SkyCondition")).cast(ByteType))
    .withColumn("sky_altitudes", extractAltitudes(col("SkyCondition")))
    .withColumn("sky_min_altitude", expr("aggregate(sky_altitudes, 1000000, (acc, x) -> IF(x < acc, x, acc))").cast(ShortType))
    .withColumn("sky_max_altitude", expr("aggregate(sky_altitudes, 0, (acc, x) -> IF(x > acc, x, acc))").cast(ShortType))
    .withColumn("sky_mean_altitude", when(size(col("sky_altitudes")) > 0, expr("aggregate(sky_altitudes, 0, (acc, x) -> acc + x) / size(sky_altitudes)")).cast(FloatType))
    .withColumn("sky_has_CB", hasCB(col("SkyCondition")).cast(BooleanType))
    .withColumn("sky_has_TCU", hasTCU(col("SkyCondition")).cast(BooleanType))

  // Ajout des colonnes binaires pour les préfixes (OVC, BKN, ...)
  val withSkyPrefixes = skyPrefixes.foldLeft(withSkyDerived) { (df, prefix) =>
    df.withColumn(s"sky_has_$prefix", col("SkyCondition").contains(prefix).cast(BooleanType))
  }

// Nettoyage colonne temporaire
  val finalSkyDF = withSkyPrefixes.drop("sky_altitudes")  
  
// Étape 5 : WeatherType → indicateurs binaires

  val withWeatherMap = finalSkyDF.withColumn("weather_scores", extractWeatherScores(col("WeatherType")))

  val weatherWeighted = weatherPairs.foldLeft(withWeatherMap) { (df, pair) =>
    df.withColumn(s"wt_score_$pair",
      coalesce(col("weather_scores").getItem(pair).cast(ByteType), lit(0.toByte))
    )
  }.drop("weather_scores")

  // Étape 6 : Casts typés
  val decimalCasts = Map(
    //"DryBulbCelsius" -> "float",
    //"WetBulbCelsius" -> "float",
    //"DewPointCelsius" -> "float",
    "StationPressure" -> "float",
    "SeaLevelPressure" -> "float",
    "HourlyPrecip" -> "float",
    "Altimeter" -> "float",
  )

  val integerCasts = Map(
    "WBAN" -> "int",
    "DryBulbFarenheit" -> "short",
    "WetBulbFarenheit" -> "short",
    "DewPointFarenheit" -> "short",
    "RelativeHumidity" -> "short",
    "WindSpeed" -> "short",
    "WindDirection" -> "short",
    "ValueForWindCharacter" -> "byte",
    "PressureTendency" -> "byte",
    "PressureChange" -> "short",
  )

  val castedDecimals = decimalCasts.foldLeft(weatherWeighted) { case (df, (colName, typ)) =>
    df.withColumn(colName, when(trim(col(colName)).isin("M", ""), null).otherwise(col(colName).cast(typ)))
  }

  val castedFinal = integerCasts.foldLeft(castedDecimals) { case (df, (colName, typ)) =>
    df.withColumn(colName, when(trim(col(colName)).isin("M", ""), null).otherwise(col(colName).cast(typ)))
  }

  // Filtrer les lignes sur WBAN présents dans airportIdWithWbanDF
  val filteredWithAirport = castedFinal
    .join(
      airportIdWithWbanDF.withColumn("WBAN", col("WBAN").cast(IntegerType)),
      Seq("WBAN"),
      "inner"
    )
    .drop("WBAN")
    .withColumn("AirportId", col("AirportId").cast(ShortType))
  // Étape 7 : Suppression des colonnes constantes
  val constantCols = Seq(
    "DryBulbFarenheitFlag", "WetBulbFarenheitFlag",
    //"WetBulbCelsiusFlag",
    "DewPointFarenheitFlag", "RelativeHumidityFlag", "StationPressureFlag",
    "PressureTendencyFlag", "PressureChangeFlag", "RecordTypeFlag"
  )

  weather_clean = filteredWithAirport.drop(constantCols: _*)
  weather_clean.printSchema()
} else {
  println("weather_clean existe déjà")
}


In [0]:
// Écrit sur un répertoire physique (meilleur contrôle)
if (!deltaWeatherCleanExists) {

  println("Création de la table delta")
  weather_clean
    .coalesce(1) // Community Edition : évite d'exploser le disque
    .write
    .mode("overwrite")
    .format("delta")
    .save(weatherCleanPath)
}


In [0]:
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.weather_clean
  USING DELTA
  LOCATION '$weatherCleanPath'
""")

In [0]:
%sql
select wcl.WeatherType, wcl.wt_score_BC,wcl.wt_score_BL,wcl.wt_score_BR
,wcl.wt_score_DR
,wcl.wt_score_DS
,wcl.wt_score_DU
,wcl.wt_score_DZ
,wcl.wt_score_FC
,wcl.wt_score_FG
,wcl.wt_score_FU
,wcl.wt_score_FZ
,wcl.wt_score_GR
,wcl.wt_score_GS
, wcl.wt_score_HZ
, wcl.wt_score_IC
, wcl.wt_score_MI
, wcl.wt_score_PL
, wcl.wt_score_PO
, wcl.wt_score_PR
, wcl.wt_score_RA
, wcl.wt_score_SA
, wcl.wt_score_SG
, wcl.wt_score_SH
, wcl.wt_score_SN
, wcl.wt_score_SQ
, wcl.wt_score_SS
, wcl.wt_score_TS
, wcl.wt_score_UP
, wcl.wt_score_VA
, wcl.wt_score_VC
from flight_project.weather_clean wcl
where wcl.WeatherType is not null and trim(wcl.WeatherType) != ''
and wcl.WeatherType like '%RA%'
--and wt_score_bl > 0